# 06 — Pathway enrichment

Equivalent to `scripts/06_pathway_enrichment.py`. Produces **Figure 5**, answers **RQ4**.

A list of 400 gene names is not an answer. "Cocaine remodels glutathione metabolism
and Toll-like receptor signalling in glia" is.

## A bug in the project guide, worth knowing about

The guide passes `gene_sets=['GO_Biological_Process_2023', 'KEGG_2021_Human']` with
`organism='fly'`. Enrichr keeps **separate library catalogues per organism**, and
those names belong to the Human catalogue. Requesting them in the Fly modality
returns nothing useful — often silently. Flagging this in your peer review of
another group would be a genuinely useful catch.

Requires internet — Enrichr is a web API.

In [ ]:
import sys

from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import scanpy as sc

import anndata as ad



import config



sc.settings.verbosity = 3

sc.settings.figdir = config.FIG_DIR

sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

sc.logging.print_header()

In [ ]:
import gseapy as gp

available = gp.get_library_name(organism=config.ENRICHR_ORGANISM)
print(f'{len(available)} Fly libraries available. Sample:')
for lib in available[:25]:
    print(' ', lib)

In [ ]:
libraries = [l for l in config.ENRICHR_LIBRARIES_PREFERRED if l in available]

for keyword in ['GO_Biological_Process', 'KEGG', 'GO_Molecular_Function']:
    if not any(keyword in c for c in libraries):
        match = [l for l in available if keyword in l]
        if match:
            libraries.append(sorted(match)[-1])

print('Using:', libraries)

## Helpers

Splitting up- from down-regulated genes matters: mixing them lets an enriched term
hide the fact that half its genes went one way and half the other, which is
biologically meaningless.

In [ ]:
def top_genes(df, direction, n=config.N_GENES_FOR_ENRICHMENT):
    d = df[(df.pvals_adj < config.PADJ_THRESHOLD) &
           (df.logfoldchanges.abs() > config.LOG2FC_THRESHOLD)]
    d = d[d.logfoldchanges > 0] if direction == 'up' else d[d.logfoldchanges < 0]
    return d.nsmallest(n, 'pvals_adj')['names'].tolist()

def run_enrichment(genes, label):
    genes = [g for g in dict.fromkeys(genes) if isinstance(g, str)]
    if len(genes) < 5:
        print(f'{label}: only {len(genes)} genes, skipping'); return None
    try:
        enr = gp.enrichr(gene_list=genes, gene_sets=libraries,
                         organism=config.ENRICHR_ORGANISM, outdir=None)
    except Exception as e:
        print(f'{label}: Enrichr failed — {e}'); return None
    res = enr.results.sort_values('Adjusted P-value')
    res.to_csv(config.TABLE_DIR / f'enrichment_{label}.csv', index=False)
    n_sig = int((res['Adjusted P-value'] < config.PADJ_THRESHOLD).sum())
    print(f'{label}: {len(genes)} genes -> {n_sig} significant terms')
    return res

def barplot(res, label, n=12):
    d = res.nsmallest(n, 'Adjusted P-value').iloc[::-1]
    if d.empty: return
    fig, ax = plt.subplots(figsize=(8, 0.36*len(d) + 1.6))
    ax.barh(range(len(d)), -np.log10(d['Adjusted P-value'].clip(lower=1e-300)),
            color='steelblue')
    ax.set_yticks(range(len(d))); ax.set_yticklabels([t[:60] for t in d['Term']], fontsize=7)
    ax.axvline(-np.log10(config.PADJ_THRESHOLD), ls='--', lw=.8, c='crimson')
    ax.set_xlabel('-log10 adjusted p'); ax.set_title(label.replace('_',' '), fontsize=10)
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / f'06_enrichment_{label}.png', dpi=150)
    plt.show()

## Global enrichment, per sex and direction

In [ ]:
for sex in ['male', 'female']:
    df = pd.read_csv(config.TABLE_DIR / f'de_global_{sex}_all.csv')
    for direction in ['up', 'down']:
        label = f'{sex}_{direction}'
        res = run_enrichment(top_genes(df, direction), label)
        if res is not None:
            display(res.head(10)[['Gene_set','Term','Adjusted P-value','Genes']])
            barplot(res, label)

## Per-cluster enrichment for the top responders

This is where the paper's most interesting findings live — their C22 surface glia
showed Notch signalling, GABA degradation, NF-κB immune pathways, and glutathione
metabolism.

In [ ]:
genes_df = pd.read_csv(config.TABLE_DIR / 'de_per_cluster_significant_genes.csv')
summary = pd.read_csv(config.TABLE_DIR / 'de_per_cluster_summary.csv')

top_clusters = (summary.dropna(subset=['n_sig'])
                .groupby('cluster')['n_sig'].sum().nlargest(4).index.tolist())
print('Top responders:', top_clusters)

In [ ]:
for cl in top_clusters:
    for sex in ['Male', 'Female']:
        sub = genes_df[(genes_df.cluster.astype(str) == str(cl)) & (genes_df.sex == sex)]
        if len(sub) < 10: continue
        genes = sub.nsmallest(config.N_GENES_FOR_ENRICHMENT, 'pvals_adj')['names'].tolist()
        safe = str(cl).replace(' ', '-').replace('/', '-')
        label = f'cluster_{safe}_{sex.lower()}'
        res = run_enrichment(genes, label)
        if res is not None:
            barplot(res, label)

## Compare with the paper

Supplemental Table S10. Terms to look for:

| Their cluster | Expected enrichment |
|---|---|
| C11, C20 — Kenyon cells | inositol phosphate metabolism, cAMP signalling |
| C16 — unannotated | GPCR signalling, serotonin and glutamate receptors, Rhodopsin-like receptors |
| C17 — astrocytes | glial metabolic support |
| C22 — surface glia | Notch, GABA degradation, NF-κB, TLR signalling, glutathione metabolism |

## A caveat for your report

Enrichr's background is all annotated fly genes, but our DE test could only ever
have found genes that survived QC and are expressed in brain. Using the full genome
as background inflates significance for brain-expressed terms. Say so; do not
present the adjusted p-values as exact.